<a href="https://colab.research.google.com/github/safdarmarwat/colab/blob/main/testing_RAG01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q gradio groq faiss-cpu sentence-transformers \
             langchain langchain-community pypdf python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [2]:
%%writefile rag_engine.py
import os, numpy as np, faiss
from sentence_transformers import SentenceTransformer
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from groq import Groq

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
faiss_index = None
chunk_store = []

def load_and_chunk(file_path, chunk_size=500, overlap=50):
    ext = os.path.splitext(file_path)[-1].lower()
    loader = PyPDFLoader(file_path) if ext == ".pdf" else TextLoader(file_path, encoding="utf-8")
    docs = loader.load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=overlap)
    return [c.page_content for c in splitter.split_documents(docs)]

def build_index(texts):
    global faiss_index, chunk_store
    chunk_store = texts
    vectors = embedder.encode(texts, show_progress_bar=True, convert_to_numpy=True).astype("float32")
    faiss_index = faiss.IndexFlatL2(vectors.shape[1])
    faiss_index.add(vectors)
    print(f"Indexed {len(texts)} chunks")

def retrieve(query, top_k=4):
    if faiss_index is None or faiss_index.ntotal == 0:
        return []
    q_vec = embedder.encode([query], convert_to_numpy=True).astype("float32")
    _, indices = faiss_index.search(q_vec, top_k)
    return [chunk_store[i] for i in indices[0] if i < len(chunk_store)]

def ask(question, chat_history, groq_api_key):
    client = Groq(api_key=groq_api_key)
    chunks = retrieve(question)
    if not chunks:
        return "No documents indexed yet. Please upload a file first."
    context = "\n\n---\n\n".join(chunks)
    prompt = f"""Answer using ONLY the context below. If not found, say so.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:"""
    messages = [{"role": "system", "content": "You are a helpful RAG assistant."}]
    for u, b in chat_history[-4:]:
        if u: messages.append({"role": "user", "content": u})
        if b: messages.append({"role": "assistant", "content": b})
    messages.append({"role": "user", "content": prompt})
    resp = client.chat.completions.create(model="llama3-8b-8192", messages=messages, temperature=0.2, max_tokens=512)
    return resp.choices[0].message.content

Writing rag_engine.py


In [3]:
%%writefile app.py
import os, gradio as gr
from rag_engine import load_and_chunk, build_index, ask

indexed = set()

def upload_file(file):
    if file is None: return "No file received."
    chunks = load_and_chunk(file.name)
    build_index(chunks)
    indexed.add(file.name)
    return f"Ready! Indexed {len(chunks)} chunks from {os.path.basename(file.name)}"

def chat(user_message, history, api_key):
    key = api_key.strip()
    if not key: return history + [[user_message, "Please enter your Groq API key."]]
    return history + [[user_message, ask(user_message, history, key)]]

with gr.Blocks(title="RAG Chatbot") as demo:
    gr.Markdown("# RAG Chatbot\nUpload a PDF or TXT, then ask questions about it.")
    with gr.Row():
        with gr.Column(scale=1):
            api_key_box = gr.Textbox(label="Groq API Key", placeholder="gsk_...", type="password")
            file_upload = gr.File(label="Upload PDF or TXT", file_types=[".pdf", ".txt"])
            upload_btn = gr.Button("Index Document", variant="primary")
            status = gr.Markdown()
            upload_btn.click(fn=upload_file, inputs=[file_upload], outputs=[status])
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(height=400)
            msg = gr.Textbox(placeholder="Ask about your document...", label="Question")
            with gr.Row():
                send = gr.Button("Send", variant="primary")
                clear = gr.Button("Clear")
            send.click(fn=chat, inputs=[msg, chatbot, api_key_box], outputs=[chatbot]).then(lambda: "", None, msg)
            msg.submit(fn=chat, inputs=[msg, chatbot, api_key_box], outputs=[chatbot]).then(lambda: "", None, msg)
            clear.click(fn=lambda: [], outputs=[chatbot])

demo.launch(share=True)

Writing app.py
